# Notebook 2 — VIX Regime Analysis

**Purpose:** using the CSVs pulled in Notebook 1, bucket each trading day into a VIX regime and compare how SPY, RSP, and TLT behave across regimes.

Run the cells in order, top to bottom.

## Setup

Mounts Drive, loads the four CSVs from Notebook 1, and merges them into one DataFrame (inner join on date — only dates where all four series have data are kept).

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

drive.mount('/content/drive')

data_dir = '/content/drive/MyDrive/MSFin_DA_Bootcamp/data'

# Load price series (SPY, RSP, TLT) and keep only Date + Close
spy = pd.read_csv(f"{data_dir}/SPY_data.csv", usecols=["Date", "Close"]).rename(columns={"Close": "SPY"})
rsp = pd.read_csv(f"{data_dir}/RSP_data.csv", usecols=["Date", "Close"]).rename(columns={"Close": "RSP"})
tlt = pd.read_csv(f"{data_dir}/TLT_data.csv", usecols=["Date", "Close"]).rename(columns={"Close": "TLT"})

# Load VIX level
vix = pd.read_csv(f"{data_dir}/VIX_data.csv").rename(columns={"date": "Date", "close": "VIX"})

# Parse dates
for df in [spy, rsp, tlt, vix]:
    df["Date"] = pd.to_datetime(df["Date"])

# Merge all four on Date (inner join — only dates present in all four are kept)
merged = spy.merge(rsp, on="Date").merge(tlt, on="Date").merge(vix, on="Date")
merged = merged.sort_values("Date").reset_index(drop=True)

print(f"✅ Merged dataset: {len(merged)} trading days, {merged['Date'].min().date()} to {merged['Date'].max().date()}")
merged.head()

## Choose your analysis window and VIX regime thresholds

Edit the variables below, then re-run this cell and everything after it.

- **Low**: VIX ≤ `vix_low`
- **Moderate**: `vix_low` < VIX ≤ `vix_moderate`
- **Elevated**: `vix_moderate` < VIX ≤ `vix_elevated`
- **High**: VIX > `vix_elevated`

In [ ]:
# --- Analysis window ---
start_date = "2006-12-31"
end_date = "2009-12-31"

# --- VIX regime thresholds ---
vix_low = 15        # Low:      VIX <= vix_low
vix_moderate = 20    # Moderate: vix_low < VIX <= vix_moderate
vix_elevated = 30    # Elevated: vix_moderate < VIX <= vix_elevated
                     # High:     VIX > vix_elevated

print(f"Window: {start_date} to {end_date}")
print(f"Thresholds: Low <= {vix_low}, Moderate <= {vix_moderate}, Elevated <= {vix_elevated}, High > {vix_elevated}")

## Filter, compute returns, and assign VIX regimes

Filters to the chosen date window, computes daily returns for SPY/RSP/TLT, and labels each day with its VIX regime based on that day's own VIX level.

In [ ]:
# Filter to the chosen window
window = merged[(merged["Date"] >= start_date) & (merged["Date"] <= end_date)].copy()

# Daily returns
for ticker in ["SPY", "RSP", "TLT"]:
    window[f"{ticker}_ret"] = window[ticker].pct_change()

window = window.dropna(subset=["SPY_ret", "RSP_ret", "TLT_ret"])

# Assign VIX regime based on that day's VIX level
def assign_regime(vix_level):
    if vix_level <= vix_low:
        return "Low"
    elif vix_level <= vix_moderate:
        return "Moderate"
    elif vix_level <= vix_elevated:
        return "Elevated"
    else:
        return "High"

window["regime"] = window["VIX"].apply(assign_regime)

regime_order = ["Low", "Moderate", "Elevated", "High"]
window["regime"] = pd.Categorical(window["regime"], categories=regime_order, ordered=True)

print("Days per regime:")
print(window["regime"].value_counts().reindex(regime_order))

## Compute metrics per regime

For each ticker and each VIX regime: average daily return, max daily return, min daily return, and annualized volatility (daily std × √252).

In [ ]:
tickers = ["SPY", "RSP", "TLT"]
metrics = {}

for metric_name, func in [
    ("Average Return", "mean"),
    ("Max Return", "max"),
    ("Min Return", "min"),
]:
    table = window.groupby("regime", observed=False)[[f"{t}_ret" for t in tickers]].agg(func)
    table.columns = tickers
    metrics[metric_name] = table.reindex(regime_order)

# Annualized volatility: daily std * sqrt(252)
vol_table = window.groupby("regime", observed=False)[[f"{t}_ret" for t in tickers]].std() * np.sqrt(252)
vol_table.columns = tickers
metrics["Annualized Volatility"] = vol_table.reindex(regime_order)

for name, table in metrics.items():
    print(f"--- {name} ---")
    print(table)
    print()

## Chart: metrics by VIX regime

A 2×2 grid — one panel per metric, VIX regimes on the x-axis, grouped bars for SPY, RSP, and TLT.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

metric_names = ["Average Return", "Max Return", "Min Return", "Annualized Volatility"]

for ax, metric_name in zip(axes, metric_names):
    table = metrics[metric_name]
    table.plot(kind="bar", ax=ax, legend=False)
    ax.set_title(metric_name)
    ax.set_xlabel("VIX Regime")
    ax.set_ylabel("Value")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.tick_params(axis="x", rotation=0)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("SPY / RSP / TLT by VIX Regime", y=1.06, fontsize=14)
fig.tight_layout()
plt.show()